# Part 4: U-Net Semantic Segmentation

把 Part 3 的影像 tile 与整数 mask 映射为逐像素类别，理解 encoder、decoder、skip connection、Dice 与 IoU。

**先修要求**：完成前序章节，了解 NumPy、张量 shape 与训练/验证划分。

## 学习目标与路线

1. 明确问题与输入输出
2. 理解核心结构
3. 运行最小实验
4. 检查指标和 shape
5. 完成练习与遥感迁移

`image [B,C,H,W] → encoder → bottleneck → decoder + skips → logits [B,K,H,W]`

In [ ]:
import numpy as np
print('NumPy:', np.__version__)

## 核心概念

- **Encoder–decoder**：编码器降低空间分辨率并增加通道；解码器恢复分辨率。
- **Skip connection**：拼接同尺度浅层细节，帮助恢复边界与小目标。
- **Logits 与 mask**：模型输出每类 logits；target 是 [B,H,W] 的 int64 类别索引。
- **Dice / IoU**：比 accuracy 更适合前景稀少的建筑、水体和道路分割。

## 最小可运行实验

先用小而确定的合成数据验证关键数学与指标。真实训练时再替换为 PyTorch 模型和 Part 3 的数据流水线。

In [ ]:
import numpy as np
rng = np.random.default_rng(4)
mask = np.zeros((64, 64), dtype=np.uint8)
mask[12:44, 18:50] = 1
score = mask * 0.75 + rng.normal(0, 0.18, mask.shape)
pred = (score > 0.5).astype(np.uint8)

def segmentation_metrics(pred, target):
    intersection = np.logical_and(pred == 1, target == 1).sum()
    union = np.logical_or(pred == 1, target == 1).sum()
    dice = 2 * intersection / ((pred == 1).sum() + (target == 1).sum() + 1e-8)
    iou = intersection / (union + 1e-8)
    return {"dice": dice, "iou": iou}

metrics = segmentation_metrics(pred, mask)
print({k: round(v, 4) for k, v in metrics.items()})
assert metrics["dice"] > metrics["iou"] > 0.8

## 结果解释

检查断言、shape 和指标是否符合预期。若最小实验失败，先修复实现；不要立即增加模型规模或下载更大数据。

## 练习

把阈值从 0.3 扫到 0.7，比较 Dice 最优阈值，并解释为什么测试集不能用于选阈值。

In [ ]:
# 答案脚手架
experiment = {'hypothesis': '', 'controlled_variables': [], 'metric': '', 'expected_failure': ''}
print(experiment)

## 常见坑与扩展

**常见坑**：skip feature 尺寸不一致；mask 被双线性插值；把 logits 先 argmax 再送入 loss；随机 tile 划分造成空间泄漏。

**扩展**：替换为真实遥感数据时，记录传感器、波段、分辨率、空间划分和随机种子。

## 完成检查

- [ ] 所有单元格从头运行
- [ ] 指标实现通过 sanity check
- [ ] 没有 train/val 空间泄漏
- [ ] 配置和随机种子已记录
- [ ] 至少分析一个失败案例